# GraphMixer-Baseline – Colab-Runner

Trainiert die **Temporal-Graph-Baseline (GraphMixer)** und bewertet sie über das
gemeinsame Eval-Modul `shared_eval`. Voraussetzung: Der Projektordner liegt in
Google Drive (oder wird hochgeladen) mit dieser Struktur:

```
Link Prediction on Hybrid Graph + Time Series Data/
├── evaluation/shared_eval.py
└── temporal graph link prediciton (GraphMixer)/
    ├── prepared/      (ml_citibike.csv/.npy/_node.npy)
    └── model/         (graphmixer*.py, train_graphmixer.py, dieses Notebook)
```

Reihenfolge: Zellen von oben nach unten ausführen.

## 1. GPU prüfen

In [ ]:
import torch
print("Torch:", torch.__version__, "| CUDA verfügbar:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Google Drive einbinden
Wenn die Dateien lokal hochgeladen wurden, kann diese Zelle übersprungen werden.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Projektpfad setzen
`ROOT` auf den Projektordner zeigen lassen. Beispielpfad anpassen.

In [ ]:
import os
# ANPASSEN: Pfad zum Projektordner in Drive
ROOT = "/content/drive/MyDrive/Link Prediction on Hybrid Graph + Time Series Data"
MODEL_DIR = os.path.join(ROOT, "temporal graph link prediciton (GraphMixer)", "model")
PREP_DIR  = os.path.join(ROOT, "temporal graph link prediciton (GraphMixer)", "prepared")
EVAL_DIR  = os.path.join(ROOT, "evaluation")

for p in [MODEL_DIR, PREP_DIR, EVAL_DIR]:
    print(("OK  " if os.path.isdir(p) else "FEHLT ") + p)
for f in ["ml_citibike.csv", "ml_citibike.npy", "ml_citibike_node.npy"]:
    fp = os.path.join(PREP_DIR, f)
    print(("OK  " if os.path.isfile(fp) else "FEHLT ") + fp)

## 4. In den model-Ordner wechseln
Die relativen Pfade in `train_graphmixer.py` erwarten dies.

In [ ]:
%cd "$MODEL_DIR" 

## 5. Konfiguration wählen
`GMConfig` mit den gewünschten Hyperparametern erzeugen. Für einen schnellen
Smoke-Test z. B. `epochs=5`; für den vollen Lauf `epochs=20` (Default).

In [ ]:
from graphmixer import GMConfig
cfg = GMConfig(epochs=20)        # z. B. GMConfig(epochs=5) für schnellen Test
print("Epochen:", cfg.epochs, "| K:", cfg.num_neighbors,
      "| Mixer-Layer:", cfg.mixer_layers, "| Batch:", cfg.batch_size)

## 6. Training + Bewertung starten
Ruft `main(cfg)` direkt auf (nutzt die GPU der Laufzeit). Gibt pro Split
AUC/AP/F1/Acc aus und schreibt die Vorhersage-CSVs nach `model/predictions/`.

In [ ]:
from train_graphmixer import main
main(cfg)

## 7. Ergebnisse nachladen / erneut bewerten
Die exportierten Vorhersagen lassen sich jederzeit erneut über `shared_eval`
bewerten – exakt dieselbe Bewertung, die später auch das Hybridmodell nutzt.

In [ ]:
import sys, pandas as pd
sys.path.insert(0, EVAL_DIR)
from shared_eval import SharedLinkEval
ev = SharedLinkEval()

for split in ["val", "test"]:
    pred = pd.read_csv(f"predictions/graphmixer_pred_{split}.csv")
    res = ev.score_binary(pred, split=split)
    print(f"[{split}] AUC={res['auc']:.3f} AP={res['ap']:.3f} "
          f"F1={res['f1']:.3f} Acc={res['accuracy']:.3f} "
          f"(n_pos={res['n_pos']}/{res['n_total']})")
pred.head()

## Nächste Schritte
- Werte notieren (AUC/AP sind die Leitmetriken für den binären Vergleich).
- Dieselben `predictions/*.csv` dienen später dem direkten Vergleich gegen den
  **Binär-Kopf des Hybridmodells** (`ev.score_binary`).
- Bei Fehlern: die letzte Fehlermeldung kopieren – dann gezielt fixen.